# Trabajo Practico Big Data

In [0]:
%restart_python

In [0]:
#importamos librerias

from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import countDistinct
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
import mlflow
from mlflow.models.signature import infer_signature
%pip install optuna
import optuna
import os
spark.conf.set("spark.databricks.execution.timeout", 3600)

### Importamos el dataset

In [0]:
#cargamos el dataset de fraudes bancarios

df = spark.table('forecast.default.data')
df.describe().show()
df.printSchema()
df.select(countDistinct("type")).show()
df.select(countDistinct("nameOrig")).show()
df.show(10)


### Procesamiento del dataset

In [0]:

# ---------------------------
# 1. Preparación de datos
# ---------------------------

df = df.withColumn("isFraud", col("isFraud").cast("double"))

indexer = StringIndexer(
    inputCol="type",
    outputCol="type_idx",
    handleInvalid="keep"
)

numeric_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
]

feature_cols = numeric_features + ["type_idx"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = indexer.fit(df).transform(df)

df_model = assembler.transform(df).select("features", "isFraud")
df_model = df_model.withColumnRenamed("isFraud", "label")

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42077651)

# ---------------------------
# 2. Evaluadores
# ---------------------------


# curva Presisión-Recall
auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)


#F2 a mano porque no está en pyspark
def compute_f2_fraud(preds, fraud_label=1.0):
    """
    Calcula F2 solo para la clase fraude (label=1),
    ignorando la clase mayoritaria que infla las métricas weighted.
    """
    total_fraud   = preds.filter(F.col("label") == fraud_label).count()
    predicted_pos = preds.filter(F.col("prediction") == fraud_label).count()
    true_pos      = preds.filter(
        (F.col("label") == fraud_label) & (F.col("prediction") == fraud_label)
    ).count()

    precision = true_pos / predicted_pos if predicted_pos > 0 else 0.0
    recall    = true_pos / total_fraud   if total_fraud   > 0 else 0.0

    denom = (4 * precision + recall)
    f2    = 5 * precision * recall / denom if denom > 0 else 0.0

    return f2, precision, recall  # devolvemos los 3 para loggear en MLflow

### Optimizacion de Hiperparametros con Optuna y Registro con MLflow

In [0]:

import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/forecast/default/prueba"
mlflow.set_experiment("/Users/martoschelp@gmail.com/RandomForest_Fraud_Optuna_Spark")

# ---------------------------
# 4. Función objetivo Optuna (multiobjetivo)
# ---------------------------

def objective(trial):
    params = {
        "numTrees": trial.suggest_int("numTrees", 50, 200, step=50),
        "maxDepth": trial.suggest_int("maxDepth", 3, 10),
        "maxBins": trial.suggest_int("maxBins", 32, 64, step=8),
        "minInstancesPerNode": trial.suggest_int("minInstancesPerNode", 1, 5)
    }

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42077651,
        **params
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    auc            = auc_eval.evaluate(preds)
    f2, prec, rec  = compute_f2_fraud(preds, fraud_label=1.0)  # 👈 clase fraude

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("pr_auc",   auc)
        mlflow.log_metric("f2_fraud",  f2)
        mlflow.log_metric("precision_fraud", prec)
        mlflow.log_metric("recall_fraud",    rec)
        mlflow.spark.log_model(model, artifact_path="RandomForest")

    return auc, f2

# ---------------------------
# 5. Optimización multiobjetivo
# ---------------------------

study = optuna.create_study(
    directions=["maximize", "maximize"]  # 👈 directions (plural) para auc y f2
)
study.optimize(objective, n_trials=20)

# ---------------------------
# 6. Ver mejores trials (Pareto front)
# ---------------------------

# Con multiobjetivo no hay un único "mejor" trial,
# sino una frontera de Pareto
pareto_trials = study.best_trials  # 👈 todos los trials no dominados

print(f"Trials en la frontera de Pareto: {len(pareto_trials)}\n")

for trial in pareto_trials:
    print(f"Trial #{trial.number}")
    print("Params:", trial.params)
    print("AUC:", trial.values[0])
    print("F2: ", trial.values[1])
    print("-" * 50)

# Opcional: si necesitás elegir UN solo trial,
# podés rankear por promedio o por la métrica prioritaria
best_by_mean = max(pareto_trials, key=lambda t: (t.values[0] + t.values[1]) / 2)
best_by_f2   = max(pareto_trials, key=lambda t: t.values[1])
best_by_auc  = max(pareto_trials, key=lambda t: t.values[0])

print("\n>> Mejor por promedio AUC+F2:")
print(f"   Trial #{best_by_mean.number} | AUC={best_by_mean.values[0]:.4f} | F2={best_by_mean.values[1]:.4f}")

print("\n>> Mejor priorizando F2:")
print(f"   Trial #{best_by_f2.number}   | AUC={best_by_f2.values[0]:.4f} | F2={best_by_f2.values[1]:.4f}")

print("\n>> Mejor priorizando AUC:")
print(f"   Trial #{best_by_auc.number}  | AUC={best_by_auc.values[0]:.4f} | F2={best_by_auc.values[1]:.4f}")

In [0]:
import optuna.visualization

# Visualización de la optimización con Optuna
optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
optuna.visualization.plot_parallel_coordinate(study)
optuna.visualization.plot_slice(study)